# 학교명 후보 개수 검증 — `data/processed/preprocessing.csv` 기반

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src"))

REPO_ROOT

In [ ]:
import pandas as pd

df = pd.read_csv(REPO_ROOT / "data" / "processed" / "preprocessing.csv", encoding="utf-8-sig")
df["comment_noun"] = df["comment_noun"].fillna("")  # 명사가 하나도 안 남은 행은 NaN으로 읽히므로 방어
df.shape

## 초/중/고/대/학교로 끝나는 명사 개수 (댓글당 학교 1개 가설 검증)

In [ ]:
from preprocessing import extract_school_candidates, noun_count

df["school_candidate"] = df["comment_noun"].apply(extract_school_candidates)
df["school_candidate_count"] = df["comment_noun"].apply(noun_count)
df[["comment", "comment_noun", "school_candidate", "school_candidate_count"]].head(20)

In [ ]:
# 가설 검증: 댓글당 학교명 후보가 정말 1개씩인지
df["school_candidate_count"].value_counts().sort_index()

In [ ]:
out_path = REPO_ROOT / "data" / "processed" / "school_candidate_counts.csv"
df[["comment_id", "comment", "comment_noun", "school_candidate", "school_candidate_count"]].to_csv(
    out_path, index=False, encoding="utf-8-sig"
)
out_path

## 1순위 ① — 후보 0건: 학교명을 아예 못 찾은 댓글

In [ ]:
zero_df = df[df["school_candidate_count"] == 0]
print(f"{len(zero_df)}개 행")
zero_df[["comment_id", "comment", "comment_noun"]]

## 1순위 ② — 후보 2건 이상: 여러 학교명 후보가 잡힌 댓글

In [ ]:
multi_df = df[df["school_candidate_count"] >= 2]
print(f"{len(multi_df)}개 행")
multi_df[["comment_id", "comment", "comment_noun", "school_candidate", "school_candidate_count"]]